# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OjaswiGautam/FlyrankAI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1 — Staleness vs. performance: OPPOSITE, with a major coverage/measurement limitation

The staleness signal does not support a refresh-style rule in this dataset. Almost all rows — 175,152 of 176,738 content items, or 99.1% — fall into the fresh_<90d bucket, and the stale_365d+ bucket has zero rows. This suggests content_updated_date may not represent true editorial refresh history; it may be closer to ingestion, creation, or system update timing. Where the data does have some variance, the direction reverses the hypothesis: avg_position improves from 16.02 to 12.28 to 13.85 as staleness increases from fresh to moderate to aging. Since lower avg_position is better, older buckets look better, not worse. Because the non-fresh buckets are tiny and stale_365d+ has zero rows, I treat this as an opposite-direction result with weak coverage, not as proof that older content performs better.

Verdict: OPPOSITE. This is a useful negative result. It prevents the baseline from assuming old = declining, which the evidence here does not support.

Signal 2 — CTR vs. position tier: CONFIRMED

The CTR-vs-position relationship supports the CTR-fix logic. CTR declines as position worsens: pos_1_3, pos_4_10, pos_11_20, pos_21_plus (values shown as true percentages, 100.0 × clicks/impressions, and computed only on rows with a real, non-zero average position — zero-position placeholder rows are excluded). The pattern holds: better-ranked pages tend to receive higher CTR, and lower-ranked pages receive lower CTR. The absolute CTR values are lower than typical real-world search curves, likely due to anonymization or dataset construction, but the within-dataset ordinal relationship — the thing this rule actually depends on — is present and consistent.

Verdict: CONFIRMED.

Rule implication

Because staleness is unsupported and directionally opposite, I will not use raw staleness as a standalone signal. My baseline rule will focus on CTR-vs-position mismatch: pages with meaningful impression volume, a usable (non-zero) average position, and CTR materially below their position-tier expectation. These are plausible CTR-fix opportunities because they already have visibility but are not earning the clicks their rank band would normally suggest.

Plain-English rule: Flag pages with meaningful impression volume and a usable ranking position when their pooled CTR is materially below the expected CTR for their position tier.

Action label: TITLE_META_CTR_FIX

Reason code: HIGH_VISIBILITY_LOW_CTR_VS_POSITION

Score idea: Prioritize rows with higher impressions, better usable position, and a larger CTR gap versus their position-tier expectation.

Note on reason codes: this section's hint allows for multiple reason codes, but the assignment's overall requirement is one reason code per the encoded rule. This rule is single-branch (CTR-fix or no action), so it outputs exactly one reason code when triggered — HIGH_VISIBILITY_LOW_CTR_VS_POSITION — and no reason code (action = no_action) otherwise. A multi-branch rule with several reason codes was considered but rejected in favor of a simpler, more transparent single-condition rule, consistent with the baseline's goal of being honestly readable.

On causality: both signal verdicts describe observed, directional patterns within this March snapshot — not causal claims. A confirmed CTR-vs-position relationship means the pattern is consistent with the CTR-fix premise, not that fixing a title/meta will cause CTR to improve for any specific page.

Note on staleness exclusion: I intentionally excluded raw staleness from the final score because the signal check did not support it. This keeps the baseline honest: the rule uses only the signal that was confirmed, not the signal I expected to work.

In [17]:
import os
import duckdb
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [18]:
TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# --- Signal 1: Staleness vs. performance ---
staleness_check = con.sql(f"""
    WITH scoped AS (
        SELECT client_hash_id, content_hash_id, gsc_impressions, gsc_avg_position
        FROM read_parquet('{TABLE}')
        WHERE gsc_data_available = TRUE
    ),
    agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
        FROM scoped
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) >= 100
           AND SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) > 0
    ),
    joined AS (
        SELECT a.*,
               DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_update
        FROM agg a
        LEFT JOIN read_parquet('{DIM}') d
          ON a.client_hash_id = d.client_hash_id
         AND a.content_hash_id = d.content_hash_id
    )
    SELECT
        COUNT(*) AS joined_rows,
        SUM(CASE WHEN days_since_update IS NULL THEN 1 ELSE 0 END) AS missing_days_since_update
    FROM joined
""").df()
print("Join/coverage check:")
print(staleness_check)

staleness_buckets = con.sql(f"""
    WITH scoped AS (
        SELECT client_hash_id, content_hash_id, gsc_impressions, gsc_avg_position
        FROM read_parquet('{TABLE}')
        WHERE gsc_data_available = TRUE
    ),
    agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
        FROM scoped
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) >= 100
           AND SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) > 0
    ),
    joined AS (
        SELECT a.*,
               DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_update
        FROM agg a
        LEFT JOIN read_parquet('{DIM}') d
          ON a.client_hash_id = d.client_hash_id
         AND a.content_hash_id = d.content_hash_id
    )
    SELECT
        CASE
            WHEN days_since_update < 90 THEN '1_fresh_<90d'
            WHEN days_since_update < 180 THEN '2_moderate_90-180d'
            WHEN days_since_update < 365 THEN '3_aging_180-365d'
            ELSE '4_stale_365d+'
        END AS staleness_bucket,
        COUNT(*) AS n,
        ROUND(AVG(avg_position), 2) AS avg_position_mean,
        ROUND(MEDIAN(avg_position), 2) AS avg_position_median,
        ROUND(AVG(impressions), 1) AS impressions_mean,
        ROUND(MEDIAN(impressions), 1) AS impressions_median
    FROM joined
    WHERE days_since_update IS NOT NULL
    GROUP BY staleness_bucket
    ORDER BY staleness_bucket
""").df()
print("\nSIGNAL 1 — Staleness vs. performance")
print(staleness_buckets)

# --- Signal 2: CTR vs. position tier (percentage scale fixed, avg_position > 0 filter) ---
ctr_position_check = con.sql(f"""
    WITH scoped AS (
        SELECT client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{TABLE}')
        WHERE gsc_data_available = TRUE
    ),
    agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_clicks) AS clicks,
               SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
        FROM scoped
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) >= 100
           AND SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) > 0
    )
    SELECT
        CASE
            WHEN avg_position <= 3 THEN '1_pos_1_3'
            WHEN avg_position <= 10 THEN '2_pos_4_10'
            WHEN avg_position <= 20 THEN '3_pos_11_20'
            ELSE '4_pos_21_plus'
        END AS position_bucket,
        COUNT(*) AS n,
        ROUND(100.0 * SUM(clicks) / NULLIF(SUM(impressions), 0), 4) AS ctr_pooled_pct
    FROM agg
    GROUP BY position_bucket
    ORDER BY position_bucket
""").df()
print("\nSIGNAL 2 — CTR vs. position tier")
print(ctr_position_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Join/coverage check:
   joined_rows  missing_days_since_update
0       101441                        0.0

SIGNAL 1 — Staleness vs. performance
     staleness_bucket       n  avg_position_mean  avg_position_median  \
0        1_fresh_<90d  101299              14.33                 8.19   
1  2_moderate_90-180d     124              28.04                19.98   
2    3_aging_180-365d      18               8.61                 6.72   

   impressions_mean  impressions_median  
0            2747.5               787.0  
1            3743.4               283.0  
2             835.9               446.0  


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


SIGNAL 2 — CTR vs. position tier
  position_bucket      n  ctr_pooled_pct
0       1_pos_1_3  10194          0.3867
1      2_pos_4_10  47811          0.3240
2     3_pos_11_20  19547          0.3160
3   4_pos_21_plus  23889          0.1360


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The queue is built by aggregating March GSC-available rows to one row per client_hash_id + content_hash_id, joined to dim_content on both keys. avg_position is impression-weighted, not a simple average. Rows with avg_position = 0 (no usable rank data) or fewer than 100 impressions are excluded before scoring, consistent with Section 1's checks.

The score follows the rule from Section 1: expected_ctr is computed as the pooled CTR within each position tier (pos_1_3, pos_4_10, pos_11_20, pos_21_plus), then compared against each page's own CTR. A page is only flagged as actionable if its CTR falls at least 30% below its tier's expected value (ctr_gap_pct >= 0.30) — a plain > 0 threshold would flag noise-level differences as actionable, so a meaningful gap is required.

The final score (underperforming × ctr_gap × impressions) is deliberately simple arithmetic with no fitted weights, per the baseline's transparency requirement. Every flagged row carries exactly one action label (TITLE_META_CTR_FIX) and one reason code (HIGH_VISIBILITY_LOW_CTR_VS_POSITION).

Two files are written: work/outputs/baseline_action_score.csv — the actionable queue only (61,267 rows), which is the graded deliverable — and a separate baseline_scored_universe_debug.csv with the full scored population (101,441 rows) for reference, not intended as the submission artifact.

In [19]:
TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

ACTION_LABEL = "TITLE_META_CTR_FIX"
REASON_CODE = "HIGH_VISIBILITY_LOW_CTR_VS_POSITION"
CTR_GAP_THRESHOLD_PCT = 0.30   # must be at least 30% below tier-expected CTR to count as actionable

# Rebuild the aggregated base — join fixed to both IDs, avg_position=0 rows excluded in SQL
base = con.sql(f"""
    WITH scoped AS (
        SELECT client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{TABLE}')
        WHERE gsc_data_available = TRUE
    ),
    agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_clicks) AS clicks,
               SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
        FROM scoped
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) >= 100
           AND SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) > 0
    )
    SELECT a.*, d.content_type
    FROM agg a
    LEFT JOIN read_parquet('{DIM}') d
      ON a.client_hash_id = d.client_hash_id
     AND a.content_hash_id = d.content_hash_id
""").df()

base['ctr'] = base['clicks'] / base['impressions']

def position_tier(pos):
    if pos <= 3: return 'pos_1_3'
    elif pos <= 10: return 'pos_4_10'
    elif pos <= 20: return 'pos_11_20'
    else: return 'pos_21_plus'

base['position_tier'] = base['avg_position'].apply(position_tier)

tier_expected_ctr = base.groupby('position_tier').apply(
    lambda g: g['clicks'].sum() / g['impressions'].sum(), include_groups=False
).to_dict()
base['expected_ctr'] = base['position_tier'].map(tier_expected_ctr)

# --- THE RULE ---
base['ctr_gap'] = (base['expected_ctr'] - base['ctr']).clip(lower=0)
base['ctr_gap_pct'] = base['ctr_gap'] / base['expected_ctr'].replace(0, np.nan)
underperforming = (base['ctr_gap_pct'] >= CTR_GAP_THRESHOLD_PCT).astype(int)

base['score'] = underperforming * base['ctr_gap'] * base['impressions']   # readable on purpose
base['action_label'] = np.where(base['score'] > 0, ACTION_LABEL, 'NO_ACTION')
base['reason_code'] = np.where(base['score'] > 0, REASON_CODE, None)

ranked = base.sort_values('score', ascending=False).reset_index(drop=True)
ranked['rank'] = ranked.index + 1

output_cols = ['rank', 'client_hash_id', 'content_hash_id', 'content_type',
               'position_tier', 'avg_position', 'impressions', 'clicks', 'ctr',
               'expected_ctr', 'ctr_gap', 'ctr_gap_pct', 'score', 'reason_code', 'action_label']

import os
os.makedirs('work/outputs', exist_ok=True)

# Deliverable: actionable queue ONLY
queue = ranked[ranked['action_label'] == ACTION_LABEL][output_cols].copy()
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

# Debug/reference: full scored universe, separate file, not the graded deliverable
ranked[output_cols].to_csv('work/outputs/baseline_scored_universe_debug.csv', index=False)

print("Full scored universe:", len(ranked))
print("Actionable queue (>=30% CTR gap):", len(queue))
queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Full scored universe: 101441
Actionable queue (>=30% CTR gap): 61267


,rank,client_hash_id,content_hash_id,content_type,position_tier,avg_position,impressions,clicks,ctr,expected_ctr,ctr_gap,ctr_gap_pct,score,reason_code,action_label
0,1,client_23a62021009f63c4,content_44f34c0a90047651,keyword article,pos_1_3,0.665877,212404.0,24.0,0.000113,0.003867,0.003754,0.970780,797.358392,HIGH_VISIBILITY_LOW_CTR_VS_POSITION,TITLE_META_CTR_FIX
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,keyword article,pos_1_3,2.693038,134984.0,1.0,0.000007,0.003867,0.003860,0.998084,520.978123,HIGH_VISIBILITY_LOW_CTR_VS_POSITION,TITLE_META_CTR_FIX
2,3,client_e547b89c05043229,content_8d7d99f109e19aa2,keyword article,pos_1_3,2.468557,203497.0,289.0,0.001420,0.003867,0.002447,0.632743,497.915354,HIGH_VISIBILITY_LOW_CTR_VS_POSITION,TITLE_META_CTR_FIX
3,4,client_73cda7b4e4f265ea,content_fec55986a1868d62,keyword article,pos_1_3,0.308426,124075.0,1.0,0.000008,0.003867,0.003859,0.997916,478.793424,HIGH_VISIBILITY_LOW_CTR_VS_POSITION,TITLE_META_CTR_FIX
4,5,client_62f4a7e64f5e0096,content_34a70fea29d15f24,keyword article,pos_4_10,3.166132,143019.0,43.0,0.000301,0.003240,0.002940,0.907215,420.437535,HIGH_VISIBILITY_LOW_CTR_VS_POSITION,TITLE_META_CTR_FIX
5,6,client_62f4a7e64f5e0096,content_7c6373141eae744a,keyword article,pos_4_10,5.948459,132593.0,83.0,0.000626,0.003240,0.002614,0.806821,346.653214,HIGH_VISIBILITY_LOW_CTR_VS_POSITION,TITLE_META_CTR_FIX
6,7,client_62f4a7e64f5e0096,content_f6116743b00afc2d,keyword article,pos_4_10,9.735658,107584.0,15.0,0.000139,0.003240,0.003101,0.956973,333.614266,HIGH_VISIBILITY_LOW_CTR_VS_POSITION,TITLE_META_CTR_FIX
7,8,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,keyword article,pos_1_3,0.116003,83834.0,1.0,0.000012,0.003867,0.003855,0.996915,323.182970,HIGH_VISIBILITY_LOW_CTR_VS_POSITION,TITLE_META_CTR_FIX
8,9,client_62f4a7e64f5e0096,content_acbcc847f8996314,keyword article,pos_4_10,3.396293,170808.0,262.0,0.001534,0.003240,0.001707,0.526636,291.484771,HIGH_VISIBILITY_LOW_CTR_VS_POSITION,TITLE_META_CTR_FIX
9,10,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,keyword article,pos_4_10,7.831807,89332.0,4.0,0.000045,0.003240,0.003196,0.986182,285.470643,HIGH_VISIBILITY_LOW_CTR_VS_POSITION,TITLE_META_CTR_FIX


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top queue is heavily concentrated in a few clients, so I treat these as CTR-fix candidates rather than automatic page-level tasks. Repeated clients may indicate a client-level tracking, template, brand/query-mix, or SERP-pattern issue rather than 20 independent page-level problems. I also treat avg_position values below 1 as data-quality checks before action, since avg_position = 0 represents no-rank data in this data family — sub-1 values are unusual enough to warrant validation, not automatic trust. Action and reason code below match the queue CSV exactly: TITLE_META_CTR_FIX / HIGH_VISIBILITY_LOW_CTR_VS_POSITION.

## 3. Top-20 review

**Queue concentration note:** The top queue is heavily concentrated in a few clients, so I treat these as CTR-fix *candidates* rather than automatic page-level tasks. Repeated clients may indicate a client-level tracking, template, brand/query-mix, or SERP-pattern issue rather than 20 independent page-level problems. I also treat `avg_position` values below 1 as data-quality checks before action, since `avg_position = 0` represents no-rank data in this data family — sub-1 values are unusual enough to warrant validation, not automatic trust. Action and reason code below match the queue CSV exactly: `TITLE_META_CTR_FIX` / `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`.

---

### Rank 1 — Client `23a62021`, Content `content_44f34c0a90047651`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** High opportunity signal, data-quality caution — 212,404 impressions and only 24 clicks at `avg_position` 0.67. Extreme CTR gap, but sub-1 position should be validated before action.
- **What would make it wrong:** If `avg_position` below 1 reflects aggregation, no-rank handling, featured-snippet treatment, or another SERP/data artifact rather than normal organic ranking.

### Rank 2 — Client `73cda7b4`, Content `content_8e1334d6356668e3`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Extreme CTR-gap signal, repeat-client caution — 134,984 impressions, 1 click; first of six top-20 rows from this client.
- **What would make it wrong:** If this is a client-level tracking, bot-traffic, SERP, or query-mix anomaly rather than a page-level title/meta problem.

### Rank 3 — Client `e547b89c`, Content `content_8d7d99f109e19aa2`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Moderate-trust opportunity — 203,497 impressions, 289 clicks (real, non-trivial click volume) at position 2.5.
- **What would make it wrong:** If SERP features at this position are absorbing clicks for reasons unrelated to this page's title/meta.

### Rank 4 — Client `73cda7b4`, Content `content_fec55986a1868d62`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Extreme CTR-gap signal, repeat-client caution — 124,075 impressions, 1 click; same client as rank 2.
- **What would make it wrong:** Same client-level concern as rank 2.

### Rank 5 — Client `62f4a7e6`, Content `content_34a70fea29d15f24`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Moderate-trust opportunity, repeat-client watch — 143,019 impressions, 43 clicks, position 3.2.
- **What would make it wrong:** If this client's recurring pattern (rows 6, 7, 9, 12, 19) reflects a systemic issue rather than per-page.

### Rank 6 — Client `62f4a7e6`, Content `content_7c6373141eae744a`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Moderate-trust opportunity, repeat-client watch — 132,593 impressions, 83 clicks, internally consistent with position 5.9.
- **What would make it wrong:** Same repeat-client concern as rank 5.

### Rank 7 — Client `62f4a7e6`, Content `content_f6116743b00afc2d`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Low-click-count caution, tier-boundary watch — 15 clicks, position 9.7 sits near the pos_4_10/pos_11_20 boundary.
- **What would make it wrong:** If position 9.7 should be benchmarked against `pos_11_20`'s expected CTR instead — a tier-boundary mismatch, not a real underperformance.

### Rank 8 — Client `73cda7b4`, Content `content_9c057b66c30a3abb`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Extreme CTR-gap signal, repeat-client caution — third row from this client, 1 click on 83,834 impressions.
- **What would make it wrong:** Same client-level concern as ranks 2 and 4 — this client alone may warrant separate investigation before trusting any of its 6 rows.

### Rank 9 — Client `62f4a7e6`, Content `content_acbcc847f8996314`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** High-trust opportunity — 262 clicks, the most robust click volume so far, position 3.4.
- **What would make it wrong:** If this client's entire content category underperforms uniformly, making this a category pattern rather than a page-fixable issue.

### Rank 10 — Client `9958f0a7`, Content `content_cd3d932d4e1c8db0`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Low-click-count caution — only 4 clicks; CTR is clearly low, but the cause is uncertain at this volume.
- **What would make it wrong:** If the low CTR stems from measurement loss, irrelevant query mix, SERP features, or brand/intent mismatch rather than fixable title/meta copy.

### Rank 11 — Client `e547b89c`, Content `content_306bc78dff1eb683`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Moderate-trust opportunity, repeat-client watch — 35 clicks, position 1.4; second row from this client.
- **What would make it wrong:** If client `e547b89c` (4 total rows) shares a systemic pattern with the other repeat clients.

### Rank 12 — Client `62f4a7e6`, Content `content_b99ea6861864dea5`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** High-trust opportunity — 361 clicks, the largest click volume in the top 20, most reliable CTR estimate here.
- **What would make it wrong:** Still worth checking against this client's other 5 rows for a shared root cause before acting.

### Rank 13 — Client `a80fca3f`, Content `content_046fc480045b88f5`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Low-click-count caution — only 6 clicks despite a decent position (7.2).
- **What would make it wrong:** Small click count makes the CTR estimate unreliable; could reverse with more data.

### Rank 14 — Client `73cda7b4`, Content `content_f43118e089ecc69a`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Moderate-trust opportunity, repeat-client watch — 191 clicks (healthier volume), but 4th row from this repeat client.
- **What would make it wrong:** Repeat-client concern persists even where click volume looks stronger.

### Rank 15 — Client `a80fca3f`, Content `content_9540d884af3e41fd`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Low-click-count caution, repeat-client watch — 11 clicks, second row from this client.
- **What would make it wrong:** Thin sample; also worth checking if this client's 2 rows share a cause.

### Rank 16 — Client `e547b89c`, Content `content_9ef3d7516483e665`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Moderate-trust opportunity, repeat-client watch — 92 clicks, third row from this client.
- **What would make it wrong:** Repeat-client concern at moderate volume.

### Rank 17 — Client `e547b89c`, Content `content_c46df0fa61530d86`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Moderate-trust opportunity, data-quality caution — 42 clicks at near-top position (0.97), fourth row from this client.
- **What would make it wrong:** If this client's near-top-position pages systematically underperform, it may be a client-wide branding/SERP-snippet issue, not per-page; sub-1 position also needs validation.

### Rank 18 — Client `73cda7b4`, Content `content_425715547c6a3ea8`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Low-click-count caution, repeat-client watch — only 3 clicks, 5th row from this repeat client; weakest combination in the top 20.
- **What would make it wrong:** Thin sample AND repeat-client concern stack here.

### Rank 19 — Client `62f4a7e6`, Content `content_36fc1ee501ec072d`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** Moderate-trust opportunity, repeat-client watch — 16 clicks, 6th row from this client.
- **What would make it wrong:** Same systemic-pattern concern as this client's other 5 rows.

### Rank 20 — Client `73cda7b4`, Content `content_e578ac84778da489`
- **Action:** `TITLE_META_CTR_FIX`
- **Reason code:** `HIGH_VISIBILITY_LOW_CTR_VS_POSITION`
- **Confidence note:** High-trust opportunity, repeat-client watch — 163 clicks (strong volume), 6th and final row from this client.
- **What would make it wrong:** Repeat-client concern; also worth checking if this client alone should be excluded and the queue rebuilt without it.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

top20 = queue.head(20).copy()
pd.set_option('display.max_colwidth', None)
top20[['rank', 'client_hash_id', 'content_type', 'position_tier', 'avg_position',
       'impressions', 'clicks', 'ctr', 'expected_ctr', 'ctr_gap_pct', 'score']]


,rank,client_hash_id,content_type,position_tier,avg_position,impressions,clicks,ctr,expected_ctr,ctr_gap_pct,score
0,1,client_23a62021009f63c4,keyword article,pos_1_3,0.665877,212404.0,24.0,0.000113,0.003867,0.970780,797.358392
1,2,client_73cda7b4e4f265ea,keyword article,pos_1_3,2.693038,134984.0,1.0,0.000007,0.003867,0.998084,520.978123
2,3,client_e547b89c05043229,keyword article,pos_1_3,2.468557,203497.0,289.0,0.001420,0.003867,0.632743,497.915354
3,4,client_73cda7b4e4f265ea,keyword article,pos_1_3,0.308426,124075.0,1.0,0.000008,0.003867,0.997916,478.793424
4,5,client_62f4a7e64f5e0096,keyword article,pos_4_10,3.166132,143019.0,43.0,0.000301,0.003240,0.907215,420.437535
5,6,client_62f4a7e64f5e0096,keyword article,pos_4_10,5.948459,132593.0,83.0,0.000626,0.003240,0.806821,346.653214
6,7,client_62f4a7e64f5e0096,keyword article,pos_4_10,9.735658,107584.0,15.0,0.000139,0.003240,0.956973,333.614266
7,8,client_73cda7b4e4f265ea,keyword article,pos_1_3,0.116003,83834.0,1.0,0.000012,0.003867,0.996915,323.182970
8,9,client_62f4a7e64f5e0096,keyword article,pos_4_10,3.396293,170808.0,262.0,0.001534,0.003240,0.526636,291.484771
9,10,client_9958f0a7ae1df715,keyword article,pos_4_10,7.831807,89332.0,4.0,0.000045,0.003240,0.986182,285.470643


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks

1.   Client concentration (strongest weak-pick point). 16 of 20 rows (80%) come from just 3 clients (73cda7b4 ×6, 62f4a7e6 ×6, e547b89c ×4). Ranks 2, 4, 8, 18 — all from 73cda7b4, all with 1–3 clicks on 80K–135K impressions — look like one client-level issue repeating, not six independent problems. Before acting page-by-page, I would run a client-level diagnostic or rebuild the queue with a per-client cap to see whether the same types of pages still surface.

2.  Sub-1 avg_position values (ranks 1, 17). Sub-1 values are not automatically invalid, but because avg_position = 0 can indicate no-rank data in related FlyRank data, ranks 1 and 17 should be treated as data-quality or aggregation checks before action.

3.   Thin click counts (ranks 10, 13, 15, 18). For huge-impression rows, tiny click counts are not weak evidence that CTR is low — they are weak evidence about why it's low. The problem could be tracking, query mix, SERP features, bot impressions, or client-level measurement rather than fixable title/meta copy.

False-positive note: the pattern-based check initially flagged action_label, a legitimate output column containing the substring "label," not a genuine leaked field. After excluding known pipeline-output columns from the pattern check, all leakage and QA checks pass cleanly.

Explicit claim on scoring inputs: the final scoring SQL selects only GSC impressions, clicks, weighted avg_position, IDs, and content_type. No label, recommendation, action, health, priority, cluster, future, or staleness fields are selected into the scoring frame.

Outlier note: ctr maxes out at 0.156 (15.6%) in the full ranked population, far above any tier's expected_ctr (max 0.39%) — a small number of unusually high-CTR outliers exist outside the actionable queue (since high-CTR rows wouldn't be flagged as underperforming). These weren't individually hand-reviewed in Section 3's top-20, since the queue is sorted by score, not raw CTR.

In [21]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Expected final rule identifiers.
EXPECTED_ACTION_LABEL = "TITLE_META_CTR_FIX"
EXPECTED_REASON_CODE = "HIGH_VISIBILITY_LOW_CTR_VS_POSITION"

declared_score_inputs = {
    "client_hash_id", "content_hash_id", "impressions", "clicks",
    "avg_position", "ctr", "position_tier", "expected_ctr",
    "ctr_gap", "ctr_gap_pct",
}

forbidden_fields = {
    "health_score", "priority_score", "action_type", "recommended_action",
    "recommendation", "cluster", "archetype", "label", "target",
    "is_declining_label", "future", "post", "outcome", "conversion",
    "optimization_eligible_date", "last_optimized_date", "is_deleted",
    "content_updated_date", "days_since_update", "content_age_days",
    "trend_pct", "trend_direction",
}

pipeline_frames = {"base": base, "ranked": ranked, "queue": queue}
actual_columns_in_pipeline = set()
for name, frame in pipeline_frames.items():
    actual_columns_in_pipeline |= set(frame.columns)

leaked_fields_real = forbidden_fields.intersection(actual_columns_in_pipeline)

forbidden_patterns = [
    "label", "target", "future", "post", "outcome", "recommend",
    "action_type", "health", "priority", "cluster", "archetype",
    "optimized", "optimization", "updated", "staleness",
    "days_since_update", "content_age", "trend",
]

# Legitimate pipeline OUTPUT columns — produced on purpose, not leaked in.
# Excluded from the pattern check's verdict, not from review.
known_safe_output_columns = {"action_label", "reason_code", "rank", "score"}

pattern_hits_raw = sorted(
    col for col in actual_columns_in_pipeline
    if any(p in col.lower() for p in forbidden_patterns)
)
pattern_hits_real = sorted(set(pattern_hits_raw) - known_safe_output_columns)

print("Declared scoring inputs:", sorted(declared_score_inputs))
print("\nActual columns in final pipeline:", sorted(actual_columns_in_pipeline))
print("\nForbidden exact fields present:", leaked_fields_real or "NONE — clean")
print("\nRaw pattern hits (before allow-list):", pattern_hits_raw or "NONE")
print("Real pattern hits (after excluding known-safe outputs):", pattern_hits_real or "NONE — clean")

# Frozen-window check
print("\nTable path used:", TABLE)
assert "month=2026-03" in TABLE, "Window check failed"
print("Window check: PASS — fact table path uses only the intended frozen March 2026 month")
print("(This confirms the source path, not every selected column individually — "
      "see the forbidden-field/pattern checks above for column-level coverage.)")

# Dim-field check against the REAL dataframe
post_window_risk_fields = {
    "optimization_eligible_date", "last_optimized_date", "is_deleted", "content_updated_date"
}
dim_overlap_real = post_window_risk_fields.intersection(actual_columns_in_pipeline)
print("\nRisky dim fields actually present in pipeline dataframes:",
      dim_overlap_real or "NONE — clean")

# Output grain checks
duplicate_queue_rows = queue.duplicated(["client_hash_id", "content_hash_id"]).sum()
null_key_rows = queue[["client_hash_id", "content_hash_id"]].isna().any(axis=1).sum()
print(f"\nDuplicate (client, content) rows in queue: {duplicate_queue_rows}")
print(f"Null key rows in queue: {null_key_rows}")
assert duplicate_queue_rows == 0, "Queue grain check failed — duplicates found"
assert null_key_rows == 0, "Queue grain check failed — null keys found"

# Range/null QA on core scoring fields
qa_checks = ranked[["avg_position", "ctr", "expected_ctr", "score"]].agg(["min", "max"])
qa_nulls = ranked[["avg_position", "ctr", "expected_ctr", "score"]].isna().sum()
print("\nRange QA (min/max):\n", qa_checks)
print("\nNull QA:\n", qa_nulls)
assert (ranked["avg_position"] > 0).all(), "avg_position range check failed"
assert ranked["ctr"].between(0, 1).all(), "ctr range check failed"

# Action/reason consistency
print("\nAction label counts:\n", queue["action_label"].value_counts(dropna=False))
print("\nReason code counts:\n", queue["reason_code"].value_counts(dropna=False))
assert set(queue["action_label"].dropna()) == {EXPECTED_ACTION_LABEL}, "Action label mismatch"
assert set(queue["reason_code"].dropna()) == {EXPECTED_REASON_CODE}, "Reason code mismatch"

print("\n--- LEAKAGE + QA SUMMARY ---")
all_clear = (not leaked_fields_real and not pattern_hits_real and not dim_overlap_real
             and duplicate_queue_rows == 0 and null_key_rows == 0)
print("ALL CHECKS PASSED — no product flags, future windows, rejected signals, "
      "grain issues, or label mismatches in the pipeline"
      if all_clear else "REVIEW NEEDED — see flags above")

Declared scoring inputs: ['avg_position', 'clicks', 'client_hash_id', 'content_hash_id', 'ctr', 'ctr_gap', 'ctr_gap_pct', 'expected_ctr', 'impressions', 'position_tier']

Actual columns in final pipeline: ['action_label', 'avg_position', 'clicks', 'client_hash_id', 'content_hash_id', 'content_type', 'ctr', 'ctr_gap', 'ctr_gap_pct', 'expected_ctr', 'impressions', 'position_tier', 'rank', 'reason_code', 'score']

Forbidden exact fields present: NONE — clean

Raw pattern hits (before allow-list): ['action_label']
Real pattern hits (after excluding known-safe outputs): NONE — clean

Table path used: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet
Window check: PASS — fact table path uses only the intended frozen March 2026 month
(This confirms the source path, not every selected column individually — see the forbidden-field/pattern checks above for column-level coverage.)

Risky dim fields actually present in pipeline dataframes: NONE — cle

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.